<p style="font-size: 30px;font-weight: bold"> Customer Segmentation for E-commerce Personalization.</p>


# Steps:

## Import required libraries:

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

## 1- Import Dataset: 
<a style="font-size: 13px;font-weight:bold;" href="https://archive.ics.uci.edu/ml/datasets/online+retail" >Online Retail Dataset from the UCI Machine Learning Repository Dataset</a>

In [ ]:
data = pd.read_excel("Online Retail.xlsx", engine="openpyxl")
data.head()

## 2. Data preparation

Keep rows with a customer identifier and exclude cancellation invoices and non-positive quantities or prices from the purchase features.

In [ ]:
data.info()

In [ ]:
data.dropna(subset=["CustomerID"], inplace=True)

# Exclude cancellations and rows that do not represent positive purchases.
is_purchase = (
    ~data["InvoiceNo"].astype(str).str.upper().str.startswith("C")
    & data["Quantity"].gt(0)
    & data["UnitPrice"].gt(0)
)
data = data.loc[is_purchase].copy()

data.isna().sum()
data.info()

## 3-Feature Engineering: 
<p style="font-size: 13px;font-weight:bold;">Extract relevant features from the dataset that capture customer behavior, such as purchase history, order frequency, total spending, etc. Calculate additional metrics if necessary, such as recency of purchase or average basket size.</p>


<p style="font-size: 12px;font-weight:bold;">Create total_amount coloumn that we will use in RFM analysis.</p>

In [ ]:
data['Total_Amount'] = data['Quantity']*data['UnitPrice']

#### Perform RFM
Recency, frequency, monetary value (RFM) is a model used in marketing analysis that segments a company’s consumer base by their purchasing patterns or habits. In particular, it evaluates customers’ recency (how long ago they made a purchase), frequency (how often they make purchases), and monetary value (how much money they spend).

#### Let’s calculate RFM values.

Calcualte the <b>M-monetary</b> value. We will be using the Total_Amount column that we have created before.

In [ ]:
m = data.groupby('CustomerID')['Total_Amount'].sum()
m = pd.DataFrame(m).reset_index()
m.head()

Calculate Frequency as the number of distinct invoices for each customer.

In [ ]:
f = data.groupby("CustomerID")["InvoiceNo"].nunique().rename("Frequency").reset_index()
f.head()

Finally, Let’s calculate <b>R-receny</b> value for each customer.
First, we need to find the when was the last purchase done in the data set.
Initially, we are changing the format and datatype of the given InvoiceDate to the format specified and then calculating the last date of purchase.

In [ ]:
data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"])
last_day = data["InvoiceDate"].max()
last_day

<b>To find out the last date of purchase of each customer</b>



In [ ]:
data['difference'] = last_day - data['InvoiceDate']
data.head()

<b>Now we need just the number of days but not the time and days attached to it just the integer. So that, it is easier to groupby later on based on each customer.So we can have a seperate function to give the integer number.</b>

In [ ]:
def get_days(x):
    y = str(x).split()[0]
    return int(y)
data['difference'] = data['difference'].apply(get_days)

<b>Now, we can groupby each customer by using CustomerId and difference column.</b>

In [ ]:
r = data.groupby('CustomerID')['difference'].min()
r = r.reset_index()
r.columns = ['CustomerID','Recency']
r.head()

<b>Now we have created all three seperate dataframes for Recency (r), frequency (f), monetary (m). Let’s group these dataframes.and perform inner join to group up 3 dataframes.</b>

In [ ]:
grouped_df = pd.merge(m, f, on = 'CustomerID',how = 'inner')
RFM_df = pd.merge(grouped_df, r, on ='CustomerID', how = 'inner')
RFM_df.columns = ['CustomerID','Monetary','Frequency','Recency']

In [ ]:
RFM_df

In [ ]:
plt.boxplot(RFM_df['Monetary'])
plt.xlabel('Monetary')
plt.show()

In [ ]:
plt.boxplot(RFM_df['Frequency'])
plt.xlabel('Frequency')
plt.show()

In [ ]:
outlier_vars = ['Monetary','Recency','Frequency']
for column in outlier_vars:
    
    lower_quartile = RFM_df[column].quantile(0.25)
    upper_quartile = RFM_df[column].quantile(0.75)
    iqr = upper_quartile - lower_quartile
    iqr_extended = iqr * 1.5
    min_border = lower_quartile - iqr_extended
    max_border = upper_quartile + iqr_extended
    
    outliers = RFM_df[(RFM_df[column] < min_border) |     (RFM_df[column] > max_border)].index
    print(f"{len(outliers)} outliers detected in column {column}")
    
    RFM_df.drop(outliers, inplace = True)

In [ ]:
plt.boxplot(RFM_df['Frequency'])
plt.xlabel('Frequency')
plt.show()

<b>A good way to get an initial understanding of the relationship between the different variables is to explore how they correlate.We calculate the correlation between our variables using corr method in the pandas library.</b>

In [ ]:
behavior_features = RFM_df[["Monetary", "Frequency", "Recency"]]
plt.figure(figsize=(9, 9))
sns.heatmap(behavior_features.corr(), annot=True, cmap="RdBu", vmin=-1, vmax=1)
plt.title("RFM Feature Correlation", fontsize=14)
plt.yticks(rotation=0)
plt.show()

<b>We see that there is a strong correlation between Monetary and Frequency. </b>

## Standardization

`StandardScaler` centers each RFM feature and scales it to unit variance. Standardization changes the feature scale; it does not make the values normally distributed.

In [ ]:
scaled_df = RFM_df[['Monetary','Frequency','Recency']]
scale_standardisation = StandardScaler()
rfm_df_scaled = scale_standardisation.fit_transform(scaled_df)
rfm_df_scaled = pd.DataFrame(rfm_df_scaled)
rfm_df_scaled.columns = ['Monetary','Frequency','Recency']
rfm_df_scaled

## 4- Dimensionality Reduction with PCA: 
<p style="font-size: 13px;font-weight:bold;">Apply PCA to reduce the dimensionality of the feature space while retaining the most informative features. This step aims to capture the underlying patterns and structure in the data. </p > 

In [ ]:
pca = PCA()
pca.fit(rfm_df_scaled)
rfm_df_scaled

<b>Now, Let’s see the explained variance ratio by each component.</b>

In [ ]:
pca.explained_variance_ratio_

After running PCA, inspect the explained-variance values and cumulative plot. Their values depend on the cleaned data and preprocessing choices.

<b>We now can plot the cumulative sum of explained variance.</b>

In [ ]:
plt.figure(figsize = (5,5))
plt.plot(range(1,4), pca.explained_variance_ratio_.cumsum(), marker = 'o', linestyle = '--')
plt.title('Explained Variance by Components')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')

The notebook retains two components for visualization and clustering. Recheck the cumulative explained variance after changing the data or preprocessing.

In [ ]:
pca = PCA(n_components =2)
pca.fit(rfm_df_scaled)
pca.components_

<b>
The result is a 2 by 3 array. We reduced our futures to two components from the original three values that explain the shape; the values themselves show the so-called loadings.

what is loading then?

Loadings are correlations between an original variable and the component.

For instance, the first value of the array shows the loading of the first feature on the first component.

Let’s put this information in a pandas data frame so that we can see them nicely. Columns are three original features and rows are two components that PCA gave us.
</b>

In [ ]:
df_pca_comp = pd.DataFrame(data = pca.components_,columns = rfm_df_scaled.columns,index = ['Component 1', 'Component 2'])
df_pca_comp

In [ ]:
plt.figure(figsize=(6,6))
sns.heatmap(df_pca_comp,
            vmin = -1, 
            vmax = 1,
            cmap = 'RdBu',
            annot = True)
plt.yticks([0, 1], 
           ['Component 1', 'Component 2' ],
           rotation = 45,
           fontsize = 12)
plt.title('Components vs Original Features',fontsize = 14)
plt.show()

<b>
We see that there is a positive correlation between Component1 and Monetary,Frequency. These are strictly related to the total spending and how many time a customer do shoping.
For the second component Recenct is by far the most prominent determinants.

Now, we have an idea about our new variables(components). We can clearly see the relationship between components and variables.

</b>

<b>Let’s transform our data and save it scores_pca.</b>

In [ ]:
# pca.transform(rfm_df_scaled)
scores_pca = pca.transform(rfm_df_scaled)
pd.DataFrame(scores_pca)

## 5- Determining Optimal Number of Clusters: 
<p style="font-size: 13px;font-weight:bold;">Use the elbow method or silhouette analysis to determine the optimal number of clusters for K-means. Experiment with different values of K and evaluate the clustering results.</p> 


In [ ]:
k_values = list(range(1, 11))
wcss_list = []
for k in k_values:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    kmeans.fit(scores_pca)
    wcss_list.append(kmeans.inertia_)

plt.plot(k_values, wcss_list, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Within-cluster sum of squares")
plt.title("Elbow curve")
plt.tight_layout()
plt.show()

The elbow curve suggests three clusters for this analysis. Treat the elbow as a heuristic and review the cluster profiles and silhouette score before interpreting the result.

## 6- K-means Clustering: 
<p style="font-size: 13px;font-weight:bold;">Perform K-means clustering on the reduced feature space. Assign each customer to a cluster based on their feature values. Analyze the resulting clusters and interpret the characteristics of each segment.</p>


In [ ]:
kmeans_pca = KMeans(n_clusters = 3, init = 'k-means++',n_init=10, random_state = 42)
kmeans_pca.fit(scores_pca)

<b>K-Means algorithm has learnt from our new components and created 3 clusters . I would like to see old datasets with new components and labels.</b>

In [ ]:
df_segm_pca_kmeans = pd.concat([RFM_df.reset_index(drop = True), pd.DataFrame(scores_pca)], axis = 1)
df_segm_pca_kmeans.columns.values[-2: ] = ['Component 1', 'Component 2']

df_segm_pca_kmeans['Segment K-means PCA'] = kmeans_pca.labels_
df_segm_pca_kmeans.head()

## 7. Cluster profiling

Summarize each cluster using the mean, minimum, and maximum Recency, Frequency, and Monetary values, plus the number of customers. Product-category analysis is not included in this notebook.

In [ ]:
RFM_df['Clusters'] = kmeans_pca.labels_
analysis = RFM_df.groupby('Clusters').agg({
    'Recency':['mean','max','min'],
    'Frequency':['mean','max','min'],
    'Monetary':['mean','max','min','count']})
analysis 

After rerunning the analysis, describe each cluster from its RFM profile. Cluster IDs are arbitrary and can change when the inputs or preprocessing change.

After rerunning, summarize the observed RFM characteristics for each cluster here. The notebook does not establish that any proposed campaign will improve retention or revenue.

Use the cluster summaries to develop testable retention or engagement hypotheses. Do not treat cluster IDs as fixed customer labels.

## 8- Visualization: 
<p style="font-size: 13px;font-weight:bold;">Visualize the clusters and their separation using scatter plots or other suitable techniques. Plot the clusters based on the reduced feature space to understand the distribution and overlap of customers.</p> 

In [ ]:
clusters = kmeans_pca.labels_
RFM = rfm_df_scaled 
RFM['labels'] = clusters
fig = plt.figure(figsize=(21,10))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(RFM["Monetary"][RFM.labels == 0], RFM["Frequency"][RFM.labels == 0], RFM["Recency"][RFM.labels == 0], c='blue', s=60)
ax.scatter(RFM["Monetary"][RFM.labels == 1],RFM["Frequency"][RFM.labels == 1], RFM["Recency"][RFM.labels == 1], c='red', s=60)
ax.scatter(RFM["Monetary"][RFM.labels == 2], RFM["Frequency"][RFM.labels == 2], RFM["Recency"][RFM.labels == 2], c='yellow', s=60)
ax.view_init(30, 185)
plt.show()

## 9- Evaluation: 
<p style="font-size: 13px;font-weight:bold;">Evaluate the quality of the clustering results using appropriate metrics such as silhouette score or within-cluster sum of squares (WCSS). Assess the cohesion and separation of the clusters to determine the effectiveness of the segmentation.</p> 

In [ ]:
silhouette = silhouette_score(scores_pca, kmeans_pca.labels_)
print(
    f"Silhouette score (k={kmeans_pca.n_clusters}, PCA space): "
    f"{silhouette:.3f}"
)

Interpret the silhouette score alongside the elbow curve and cluster profiles. It measures cohesion and separation in the PCA feature space; it does not measure business impact.

## 10- Personalization and Recommendations: 
<p style="font-size: 13px;font-weight:bold;">Based on the identified customer segments, develop personalized marketing strategies and recommendations. Tailor promotions, product suggestions, or communication channels for each cluster to enhance customer engagement and satisfaction. Interpretation and Insights: Interpret the results and provide insights about the different customer segments. Discuss the implications for the e-commerce business, such as targeted marketing, customer retention, or inventory management. Remember to adhere to ethical guidelines and data privacy regulations while working with customer data.</p>

After rerunning, summarize the observed RFM characteristics for each cluster here. The notebook does not establish that any proposed campaign will improve retention or revenue.